# Clustering Experiments — CustomerInsight AI

## Research Objective

Compare K-Means, Agglomerative Clustering and DBSCAN using customer-level RFM measurements.

This notebook uses existing project code instead of implementing clustering algorithms from scratch.

## 1. Research Question

**Which of the tested clustering configurations produces the strongest internal separation?**

We use silhouette score and cluster coverage, while recognising their limitations.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt

# Make the project package importable in Jupyter.
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if not (ROOT / 'src/customerinsight').exists():
    raise FileNotFoundError('Open this notebook from the project.')
sys.path.insert(0, str(ROOT / 'src'))

from customerinsight.clustering import (
    prepare_clustering_features,
    run_clustering_experiments,
)

print('Project:', ROOT)

## 2. Research Data

We use the saved customer-level RFM dataset. Each row represents one customer.

- Recency: days since the last qualifying purchase
- Frequency: number of distinct positive-sale invoices
- Monetary: gross positive spending

Historical returns and cancellations are not fully reconciled into net monetary value.

In [ ]:
rfm_path = ROOT / 'data/processed/customer_rfm.csv'
rfm = pd.read_csv(rfm_path)

required = {'customer_id', 'recency', 'frequency', 'monetary'}
assert required.issubset(rfm.columns)
assert not rfm.empty

print('Customers:', len(rfm))
display(rfm.head())
display(rfm[['recency', 'frequency', 'monetary']].describe())

## 3. Feature Preparation

The existing project implementation prepares RFM features for clustering using the methodology implemented in `clustering.py`.

We reuse that function instead of independently recreating its preprocessing steps.

In [ ]:
features = prepare_clustering_features(rfm)
print('Prepared feature type:', type(features).__name__)
print('Prepared shape:', features.shape)
display(features.head() if hasattr(features, 'head') else features[:5])

## 4. Saved Baseline Evidence

First, inspect the original experiment results.

Reading this CSV does **not** reproduce the experiment; it establishes the saved baseline against which future execution can be checked.

In [ ]:
baseline_path = ROOT / 'results/experiments/clustering_baseline.csv'
baseline = pd.read_csv(baseline_path)
display(baseline)

ax = baseline.plot.bar(
    x='algorithm', y='silhouette', legend=False,
    title='Baseline clustering comparison', figsize=(8, 4)
)
ax.set_ylabel('Silhouette score')
ax.set_xlabel('Algorithm')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 5. Re-execution of Clustering Experiments

The following cell calls the same project function used to create the original clustering comparison.

It does not explicitly save any results. Before executing this cell, confirm that `run_clustering_experiments` itself does not write to protected experiment files.

In [ ]:
# Safety guard: intentionally disabled until function
# side effects and execution settings are reviewed.
RUN_EXPERIMENT = False

if RUN_EXPERIMENT:
    reproduced = run_clustering_experiments(features)
    print('Experiment result type:', type(reproduced).__name__)
    display(reproduced)
else:
    print('Experiment not run. Review the function first.')

## 6. Verified Baseline Results

The saved Phase 04 results were:

| Algorithm | Clusters | Silhouette | Coverage |
|---|---:|---:|---:|
| K-Means | 4 | 0.332372 | 1.0000 |
| Agglomerative | 4 | 0.270554 | 1.0000 |
| DBSCAN | 2 | 0.289804 | 0.9818 |

These values are the recorded baseline results, not a claim that the notebook has reproduced them.

## 7. Interpretation

K-Means achieved the highest silhouette score among the three tested baseline configurations.

However, this result does not establish that K-Means is universally superior or that its clusters are ground-truth customer categories.

## 8. Limitations

- Only selected algorithm configurations were compared.
- Silhouette evaluates internal structure, not business value.
- DBSCAN assigned some observations to noise.
- Results depend on feature transformations and parameters.
- A fresh experiment execution must be compared with the original saved results before reproducibility is claimed.

## 9. Reproducibility Status

**Notebook restored; numerical reproduction pending.**

Next actions:

1. Review clustering function side effects.
2. Execute the experiment under controlled conditions.
3. Compare regenerated scores and cluster counts.
4. Record discrepancies and software versions.